# Unit 08 - Peeking and Sequential Testing (Exercise) · **V2 material**

**Atoms:** `U08-A7` · **Runtime:** ~25 seconds

## Without code

Naive stop rate well above 0.05. The always-valid rate lands **at or below** 0.05 - usually far below, because this boundary is deliberately conservative. That conservatism is the price of being allowed to look every day.

## 1. The question

Implement naive and always-valid stopping rules on simulated A/A tests.

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 4. TODO - naive peeking

300 simulations, 10 daily looks, 150 users per arm per day, `p=0.15`. Return `naive_rate` = fraction stopped with any daily `p < 0.05`.

In [ ]:
naive_rate = None  # TODO
assert naive_rate is not None
print('Naive rate:', round(naive_rate, 3))
assert naive_rate > 0.10

## 5. TODO - always-valid boundary

Implement `msprt_boundary(n, alpha=0.05)` and `simulate_always_valid_stop(n_days, n_per_arm, p)`. Return `av_rate` over 300 simulations.

In [ ]:
av_rate = None  # TODO
assert av_rate is not None
print('Always-valid rate:', round(av_rate, 3))
assert av_rate <= 0.05

## 6. TODO - compare

Assert `naive_rate > av_rate`.

In [ ]:
# TODO
assert naive_rate > av_rate
if av_rate > 0:
    print('Inflation factor:', round(naive_rate / av_rate, 2))
else:
    print('The always-valid rule never stopped - the inflation factor is off the scale')

**Takeaway:** Choose your boundary before you peek. **Unit:** [V2 unit 08](../V2/units/unit-08-power-duration-sample-size/README.md)

## Hints

Use `stats.ttest_ind_from_stats` on cumulative counts each day. Boundary: `sqrt(2*log(1/alpha) + log(n))`.

## Spoiler

```python
n_days, n_per_arm_day, p0, n_sims, alpha = 10, 150, 0.15, 300, 0.05

def cumulative_z(n_days, n_per_arm, p):
    """Yield the z statistic after each daily look at an A/A test."""
    cum_c = cum_t = n_c = n_t = 0
    for _ in range(n_days):
        cum_c += np.random.binomial(1, p, n_per_arm).sum()
        cum_t += np.random.binomial(1, p, n_per_arm).sum()
        n_c += n_per_arm
        n_t += n_per_arm
        pc, pt = cum_c / n_c, cum_t / n_t
        se = np.sqrt(pt * (1 - pt) / n_t + pc * (1 - pc) / n_c)
        yield ((pt - pc) / se if se > 0 else 0.0), n_c + n_t

def naive_stops(n_days, n_per_arm, p, alpha=0.05):
    return any(2 * (1 - stats.norm.cdf(abs(z))) < alpha
               for z, _ in cumulative_z(n_days, n_per_arm, p))

naive_rate = np.mean([naive_stops(n_days, n_per_arm_day, p0) for _ in range(n_sims)])

def msprt_boundary(n_total, alpha=0.05):
    # Threshold on |z| that grows with sample size, so repeated looks stay honest
    return np.sqrt(2 * np.log(1 / alpha) + np.log(max(n_total, 2)))

def simulate_always_valid_stop(n_days, n_per_arm, p, alpha=0.05):
    return any(abs(z) > msprt_boundary(n_total, alpha)
               for z, n_total in cumulative_z(n_days, n_per_arm, p))

av_rate = np.mean([simulate_always_valid_stop(n_days, n_per_arm_day, p0)
                   for _ in range(n_sims)])
```